In [14]:
%%bash
# Download habitat test scene + pointnav dataset (needed for this tutorial)
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv

if [ ! -d /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_scenes ]; then
    python -m habitat_sim.utils.datasets_download \
        --uids habitat_test_scenes \
        --data-path /content/drive/MyDrive/HabitatData \
        --no-replace --no-prune 2>&1 | tail -5
fi

if [ ! -d /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_pointnav_dataset ]; then
    python -m habitat_sim.utils.datasets_download \
        --uids habitat_test_pointnav_dataset \
        --data-path /content/drive/MyDrive/HabitatData \
        --no-replace --no-prune 2>&1 | tail -5
fi

echo "Done"

    subprocess.check_call(split_command)
  File "/opt/conda/envs/habitatEnv/lib/python3.9/subprocess.py", line 373, in check_call
    raise CalledProcessError(retcode, cmd)
subprocess.CalledProcessError: Command '['git', 'clone', '--depth', '1', '--branch', 'main', 'https://huggingface.co/datasets/ai-habitat/habitat_test_scenes.git', '/content/drive/MyDrive/HabitatData/versioned_data/habitat_test_scenes']' returned non-zero exit status 255.
git clone --depth 1 --branch main https://huggingface.co/datasets/ai-habitat/habitat_test_scenes.git /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_scenes
  File "/opt/conda/envs/habitatEnv/lib/python3.9/site-packages/habitat_sim-0.3.3-py3.9-linux-x86_64.egg/habitat_sim/utils/datasets_download.py", line 953, in main
    download_and_place(
  File "/opt/conda/envs/habitatEnv/lib/python3.9/site-packages/habitat_sim-0.3.3-py3.9-linux-x86_64.egg/habitat_sim/utils/datasets_download.py", line 813, in download_and_place
    os.symlink(src=v

In [ ]:
%%bash
# Clean up any previous symlinks that might conflict
# rm -f /content/data/scene_datasets /content/data/datasets

# Create parent dirs
mkdir -p /content/data/datasets/pointnav
mkdir -p /content/data/scene_datasets

# Pointnav dataset symlink
ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_pointnav_dataset_1.0 \
    /content/data/datasets/pointnav/habitat-test-scenes

# Scene datasets symlink  
ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_scenes \
    /content/data/scene_datasets/habitat-test-scenes

# Verify
ls /content/data/datasets/pointnav/habitat-test-scenes/v1/train/train.json.gz 2>/dev/null && echo "Pointnav dataset OK"
ls /content/data/scene_datasets/habitat-test-scenes/ 2>/dev/null | head -5


/content/data/datasets/pointnav/habitat-test-scenes/v1/train/train.json.gz
Pointnav dataset OK
apartment_1.glb
apartment_1.navmesh
README.md
skokloster-castle.glb
skokloster-castle.navmesh


In [ ]:
# %%bash
# find /content/drive/MyDrive/HabitatData -name "train.json.gz" 2>/dev/null
# find /content/drive/MyDrive/HabitatData -type d -name "pointnav" 2>/dev/null

Process is terminated.


In [4]:
%%bash
# EGL config for headless rendering
cat > /tmp/nvidia_egl.json << 'EOF'
{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
EOF
echo "EGL ready"

EGL ready


## 2. Helper utilities — display sample observations

In [12]:
%%bash
cat > /content/habitat_lab_utils.py << 'PYEOF'
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
import habitat
from habitat.core.logging import logger
from habitat.core.registry import registry
from habitat.sims.habitat_simulator.actions import HabitatSimActions
from habitat.tasks.nav.nav import NavigationTask
from habitat.config.default_structured_configs import LabSensorConfig
from gym import spaces

CONFIG_PATH = "/content/habitat-lab-v033/habitat-lab/habitat/config/benchmark/nav/pointnav/pointnav_habitat_test.yaml"

def display_sample(rgb_obs, semantic_obs=None, depth_obs=None, save_path="/content/obs.png"):
    from habitat_sim.utils.common import d3_40_colors_rgb
    rgb_img = Image.fromarray(rgb_obs, mode="RGB")
    arr = [rgb_img]
    titles = ["rgb"]
    if semantic_obs is not None and semantic_obs.size != 0:
        sem_img = Image.new("P", (semantic_obs.shape[1], semantic_obs.shape[0]))
        sem_img.putpalette(d3_40_colors_rgb.flatten())
        sem_img.putdata((semantic_obs.flatten() % 40).astype(np.uint8))
        sem_img = sem_img.convert("RGBA")
        arr.append(sem_img)
        titles.append("semantic")
    if depth_obs is not None and depth_obs.size != 0:
        depth_img = Image.fromarray((depth_obs / 10 * 255).astype(np.uint8), mode="L")
        arr.append(depth_img)
        titles.append("depth")
    fig = plt.figure(figsize=(12, 8))
    for i, data in enumerate(arr):
        ax = plt.subplot(1, len(arr), i + 1)
        ax.axis("off")
        ax.set_title(titles[i])
        plt.imshow(data)
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)

print("habitat_lab_utils.py ready")
PYEOF

## 3. Run PointNav task


In [21]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
from habitat_lab_utils import *
import imageio

config = habitat.get_config(
    config_path=CONFIG_PATH,
    overrides=[
        "habitat.environment.max_episode_steps=10",
        "habitat.environment.iterator_options.shuffle=False",
    ],
)

env = habitat.Env(config=config)
obs = env.reset()
print("Observation keys:", list(obs.keys()))

valid_actions = ["turn_left", "turn_right", "move_forward", "stop"]
frames = []
action = None

while action != "stop":
    frames.append(obs["rgb"])
    dist, angle = obs["pointgoal_with_gps_compass"]
    print(f"dist={dist:.2f}m, angle={angle:.2f} rad")
    action = valid_actions.pop(0)
    obs = env.step({"action": action})

print("Metrics:", env.get_metrics())
env.close()

# Save as video
w = imageio.get_writer("/content/pointnav_demo.mp4", fps=2)
for f in frames:
    w.append_data(f)
w.close()
print(f"Saved {len(frames)} frames")
PY

Renderer: Tesla T4/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 580.82.07
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
habitat_lab_utils.py ready
Observation keys: ['rgb', 'depth', 'poin

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 04:06:40,366 Initializing dataset PointNav-v1
2026-04-18 04:06:40,862 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

In [22]:
from IPython.display import Video, display
display(Video("/content/pointnav_demo.mp4", embed=True, width=500))

## 5. Create a custom Task — TestNav-v0


In [28]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
from habitat_lab_utils import *

@registry.register_task(name="TestNav-v0")
class NewNavigationTask(NavigationTask):
    def __init__(self, config, sim, dataset):
        logger.info("Creating a new task type")
        super().__init__(config=config, sim=sim, dataset=dataset)

    def _check_episode_is_active(self, *args, **kwargs):
        collision = self._sim.previous_step_collided
        stop_called = not getattr(self, "is_stop_called", False)
        # Episode ends on collision OR when stop is called
        return collision or stop_called

config = habitat.get_config(
    config_path=CONFIG_PATH,
    overrides=[
        "habitat.environment.max_episode_steps=10",
        "habitat.environment.iterator_options.shuffle=False",
    ],
)
with habitat.config.read_write(config):
    config.habitat.task.type = "TestNav-v0"

env = habitat.Env(config=config)
obs = env.reset()
valid_actions = ["turn_left", "turn_right", "move_forward", "stop"]

steps = 0
while not env.episode_over:
    action = valid_actions.pop(0)
    obs = env.step({"action": action, "action_args": None})
    steps += 1
    print(f"step {steps}, episode_over={env.episode_over}")

print(f"Done after {steps} steps")
env.close()
PY

Renderer: Tesla T4/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 580.82.07
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
habitat_lab_utils.py ready
step 1, episode_over=False
step 2, episo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 05:10:17,465 Initializing dataset PointNav-v1
2026-04-18 05:10:17,768 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

## 6. Create a custom Sensor — agent_position


In [25]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
from habitat_lab_utils import *

@registry.register_sensor(name="agent_position_sensor")
class AgentPositionSensor(habitat.Sensor):
    def __init__(self, sim, config, **kwargs):
        super().__init__(config=config)
        self._sim = sim

    def _get_uuid(self, *args, **kwargs):
        return "agent_position"

    def _get_sensor_type(self, *args, **kwargs):
        return habitat.SensorTypes.POSITION

    def _get_observation_space(self, *args, **kwargs):
        return spaces.Box(
            low=np.finfo(np.float32).min,
            high=np.finfo(np.float32).max,
            shape=(3,),
            dtype=np.float32,
        )

    def get_observation(self, observations, *args, episode, **kwargs):
        return self._sim.get_agent_state().position

config = habitat.get_config(
    config_path=CONFIG_PATH,
    overrides=[
        "habitat.environment.max_episode_steps=10",
        "habitat.environment.iterator_options.shuffle=False",
    ],
)
with habitat.config.read_write(config):
    config.habitat.task.lab_sensors["agent_position_sensor"] = LabSensorConfig(type="agent_position_sensor")

env = habitat.Env(config=config)
obs = env.reset()
print("Observation keys:", list(obs.keys()))
print("Agent position:", obs["agent_position"])

# Take a few steps and print position each time
valid_actions = ["turn_left", "move_forward", "move_forward", "turn_right"]
for action in valid_actions:
    obs = env.step({"action": action, "action_args": None})
    print(f"After {action}: position={obs['agent_position']}")

env.close()
PY

Renderer: Tesla T4/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 580.82.07
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
habitat_lab_utils.py ready
Observation keys: ['rgb', 'depth', 'poin

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 04:15:53,029 Initializing dataset PointNav-v1
2026-04-18 04:15:53,543 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

## 7. Create a custom Agent — ForwardOnlyAgent


In [26]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
from habitat_lab_utils import *
import imageio

class ForwardOnlyAgent(habitat.Agent):
    def __init__(self, success_distance, goal_sensor_uuid):
        self.dist_threshold_to_stop = success_distance
        self.goal_sensor_uuid = goal_sensor_uuid

    def reset(self):
        pass

    def is_goal_reached(self, observations):
        return observations[self.goal_sensor_uuid][0] <= self.dist_threshold_to_stop

    def act(self, observations):
        if self.is_goal_reached(observations):
            return {"action": HabitatSimActions.stop}
        return {"action": HabitatSimActions.move_forward}

config = habitat.get_config(
    config_path=CONFIG_PATH,
    overrides=[
        "habitat.environment.max_episode_steps=200",
        "habitat.environment.iterator_options.shuffle=False",
    ],
)

env = habitat.Env(config=config)
agent = ForwardOnlyAgent(success_distance=0.2, goal_sensor_uuid="pointgoal_with_gps_compass")

obs = env.reset()
agent.reset()

frames = []
for step in range(200):
    frames.append(obs["rgb"])
    action = agent.act(obs)
    obs = env.step(action)
    if env.episode_over:
        print(f"Episode done at step {step}")
        break

print("Metrics:", env.get_metrics())
env.close()

w = imageio.get_writer("/content/forward_agent.mp4", fps=10)
for f in frames:
    w.append_data(f)
w.close()
print(f"Saved {len(frames)} frames")
PY

Renderer: Tesla T4/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 580.82.07
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
habitat_lab_utils.py ready
Episode done at step 199
Metrics: {'dist

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 04:18:51,798 Initializing dataset PointNav-v1
2026-04-18 04:18:52,087 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

In [27]:
from IPython.display import Video, display
display(Video("/content/forward_agent.mp4", embed=True, width=500))